In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

dataset_path = r"C:\Users\user\Documents\E-Commerce-Customer-Business-Analytics\data\processed"

orders = pd.read_csv(dataset_path + r"\orders_cleaned.csv")

orders["order_date"] = pd.to_datetime(orders["order_date"])

print("Cohort Analysis Ready!")
print("Total Orders:", len(orders))

Cohort Analysis Ready!
Total Orders: 100000


In [2]:
orders["Order_Month"] = orders["order_date"].dt.to_period("M")

orders["Cohort_Month"] = orders.groupby("customer_id")["order_date"].transform("min").dt.to_period("M")

print(orders[["customer_id", "Order_Month", "Cohort_Month"]].head(10))

  customer_id Order_Month Cohort_Month
0  CUST103868     2025-08      2025-04
1  CUST118472     2023-02      2023-01
2  CUST120793     2024-12      2024-04
3  CUST109166     2025-11      2025-09
4  CUST119895     2025-07      2025-06
5  CUST123408     2025-01      2024-05
6  CUST117355     2025-05      2024-06
7  CUST115056     2025-10      2023-11
8  CUST107870     2023-07      2023-03
9  CUST117396     2024-05      2024-05


In [3]:
orders["Cohort_Index"] = (
    (orders["Order_Month"].dt.year - orders["Cohort_Month"].dt.year) * 12
    + (orders["Order_Month"].dt.month - orders["Cohort_Month"].dt.month)
)

print(orders[["customer_id", "Order_Month", "Cohort_Month", "Cohort_Index"]].head(10))

  customer_id Order_Month Cohort_Month  Cohort_Index
0  CUST103868     2025-08      2025-04             4
1  CUST118472     2023-02      2023-01             1
2  CUST120793     2024-12      2024-04             8
3  CUST109166     2025-11      2025-09             2
4  CUST119895     2025-07      2025-06             1
5  CUST123408     2025-01      2024-05             8
6  CUST117355     2025-05      2024-06            11
7  CUST115056     2025-10      2023-11            23
8  CUST107870     2023-07      2023-03             4
9  CUST117396     2024-05      2024-05             0


In [4]:
cohort_data = orders.groupby(
    ["Cohort_Month", "Cohort_Index"]
)["customer_id"].nunique().reset_index()

cohort_table = cohort_data.pivot(
    index="Cohort_Month",
    columns="Cohort_Index",
    values="customer_id"
)

print(cohort_table)

Cohort_Index      0      1      2      3      4      5      6      7      8   \
Cohort_Month                                                                   
2023-01        248.0  147.0  116.0   65.0   44.0   30.0   30.0   20.0   16.0   
2023-02        439.0  212.0  171.0  125.0   95.0   84.0   63.0   53.0   52.0   
2023-03        715.0  305.0  254.0  182.0  153.0  133.0   96.0  110.0   84.0   
2023-04        785.0  309.0  247.0  176.0  161.0  124.0  126.0  114.0   92.0   
2023-05        817.0  299.0  253.0  199.0  155.0  153.0  140.0  123.0   94.0   
2023-06        900.0  338.0  286.0  227.0  229.0  204.0  157.0  121.0   96.0   
2023-07        881.0  300.0  253.0  233.0  222.0  153.0  111.0   86.0   87.0   
2023-08        778.0  236.0  193.0  239.0  164.0  129.0  125.0  101.0   78.0   
2023-09        620.0  179.0  194.0  142.0  114.0   79.0   87.0   72.0   59.0   
2023-10        673.0  234.0  174.0  150.0   98.0  112.0  112.0   87.0   72.0   
2023-11        753.0  223.0  179.0  133.

In [5]:
cohort_sizes = cohort_table.iloc[:, 0]

retention_table = cohort_table.divide(cohort_sizes, axis=0) * 100

retention_table = retention_table.round(2)

print("COHORT RETENTION RATE (%):")
print(retention_table)

COHORT RETENTION RATE (%):
Cohort_Index     0      1      2      3      4      5      6      7      8   \
Cohort_Month                                                                  
2023-01       100.0  59.27  46.77  26.21  17.74  12.10  12.10   8.06   6.45   
2023-02       100.0  48.29  38.95  28.47  21.64  19.13  14.35  12.07  11.85   
2023-03       100.0  42.66  35.52  25.45  21.40  18.60  13.43  15.38  11.75   
2023-04       100.0  39.36  31.46  22.42  20.51  15.80  16.05  14.52  11.72   
2023-05       100.0  36.60  30.97  24.36  18.97  18.73  17.14  15.06  11.51   
2023-06       100.0  37.56  31.78  25.22  25.44  22.67  17.44  13.44  10.67   
2023-07       100.0  34.05  28.72  26.45  25.20  17.37  12.60   9.76   9.88   
2023-08       100.0  30.33  24.81  30.72  21.08  16.58  16.07  12.98  10.03   
2023-09       100.0  28.87  31.29  22.90  18.39  12.74  14.03  11.61   9.52   
2023-10       100.0  34.77  25.85  22.29  14.56  16.64  16.64  12.93  10.70   
2023-11       100.0  29.6

In [9]:
overall_retention = retention_table.mean(axis=0).round(2)

print("AVERAGE CUSTOMER RETENTION:")
print(overall_retention)

AVERAGE CUSTOMER RETENTION:
Cohort_Index
0     100.00
1      36.24
2      30.57
3      25.26
4      21.05
5      18.12
6      16.68
7      13.70
8      12.24
9      10.97
10      9.44
11      8.61
12      7.56
13      7.00
14      5.86
15      5.51
16      5.20
17      4.53
18      4.38
19      3.59
20      3.28
21      2.99
22      3.12
23      2.66
24      2.19
25      1.91
26      1.72
27      1.92
28      1.33
29      1.28
30      0.99
31      1.37
32      1.02
33      0.92
34      0.68
35      0.40
dtype: float64


In [10]:
cohort_table.to_csv(dataset_path + r"\cohort_analysis.csv")
retention_table.to_csv(dataset_path + r"\cohort_retention_analysis.csv")

print("Cohort & Retention Results Saved Successfully!")

Cohort & Retention Results Saved Successfully!
